# OpenAlex Enrichment Script

This notebook enriches the arXiv dataset with OpenAlex metadata to support recommendation experiments.

It performs DOI-first matching, falls back to title-based matching, and adds citation-related fields to each paper record.

Outputs:
- `data/openalex_enriched.jsonl`
- `data/openalex_enrichment_metadata.json`

Note: API failures are handled gracefully; records that still fail after retries are skipped and flagged.

In [ ]:
import json
import time
import re
from datetime import datetime
from typing import Dict, List, Optional, Tuple
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
from difflib import SequenceMatcher

OPENALEX_BASE = "https://api.openalex.org"

# Replace with your actual OpenAlex API key
API_KEY = "YOUR_OPENALEX_API_KEY"
EMAIL = "youremail@example.com"

INPUT_JSONL = "data/papers.jsonl"
OUTPUT_JSONL = "data/openalex_enriched.jsonl"
OUTPUT_META = "data/openalex_enrichment_metadata.json"

DOI_BATCH_SIZE = 100
REQUEST_SLEEP_SECONDS = 0.5
MAX_RETRIES = 5
TITLE_MATCH_THRESHOLD = 0.80


def load_jsonl(filepath: str) -> List[Dict]:
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def write_jsonl(records: List[Dict], filepath: str) -> None:
    with open(filepath, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def build_headers() -> Dict[str, str]:
    user_agent = "AyomideMScProject/1.0"
    if EMAIL:
        user_agent += f" ({EMAIL})"
    return {"User-Agent": user_agent}


def fetch_json(url: str, max_retries: int = MAX_RETRIES) -> Dict:
    headers = build_headers()

    for attempt in range(max_retries):
        try:
            req = Request(url, headers=headers)
            with urlopen(req, timeout=60) as response:
                return json.loads(response.read().decode("utf-8"))

        except HTTPError as e:
            if e.code in (429, 500, 502, 503, 504):
                sleep_time = 2 ** attempt
                print(f"Retryable HTTP error {e.code}. Sleeping {sleep_time}s.")
                time.sleep(sleep_time)
                continue
            raise

        except URLError:
            sleep_time = 2 ** attempt
            print(f"Network error. Sleeping {sleep_time}s.")
            time.sleep(sleep_time)
            continue

    raise RuntimeError(f"Failed after {max_retries} retries: {url}")


def build_openalex_url(path: str, params: Dict[str, str]) -> str:
    params = dict(params)
    params["api_key"] = API_KEY
    return f"{OPENALEX_BASE}{path}?{urlencode(params)}"


def strip_doi_prefix(doi: str) -> str:
    doi = doi.strip()
    doi = doi.replace("https://doi.org/", "")
    doi = doi.replace("http://doi.org/", "")
    doi = doi.replace("doi.org/", "")
    return doi.lower()


def normalise_title_for_match(title: Optional[str]) -> str:
    if not title:
        return ""

    title = title.lower().strip()
    title = re.sub(r"\s+", " ", title)
    title = re.sub(r"[^a-z0-9\s]", "", title)
    return title


def title_similarity(a: str, b: str) -> float:
    a_norm = normalise_title_for_match(a)
    b_norm = normalise_title_for_match(b)
    return SequenceMatcher(None, a_norm, b_norm).ratio()


def bulk_lookup_by_doi(dois: List[str]) -> Dict[str, Dict]:
    """
    Returns mapping:
    cleaned_doi -> OpenAlex work record
    """
    clean_dois = [strip_doi_prefix(d) for d in dois if d]
    if not clean_dois:
        return {}

    filter_value = "doi:" + "|".join(clean_dois)
    params = {
        "filter": filter_value,
        "per_page": "100",
        "select": "id,doi,title,publication_year,referenced_works,cited_by_count",
    }
    url = build_openalex_url("/works", params)
    data = fetch_json(url)

    results = {}
    for item in data.get("results", []):
        doi = item.get("doi")
        if doi:
            results[strip_doi_prefix(doi)] = item

    return results


def search_by_title(title: Optional[str], year: Optional[int] = None, min_similarity: float = TITLE_MATCH_THRESHOLD) -> Optional[Dict]:
    """
    Fallback title-based search. Returns best candidate only if similarity is above threshold.
    """
    if not title:
        return None

    params = {
        "search": title,
        "per_page": "10",
        "select": "id,doi,title,publication_year,referenced_works,cited_by_count",
    }
    url = build_openalex_url("/works", params)
    data = fetch_json(url)

    candidates = data.get("results", [])
    if not candidates:
        return None

    best_candidate = None
    best_score = 0.0

    for cand in candidates:
        cand_title = cand.get("title", "")
        score = title_similarity(title, cand_title)

        if year is not None and cand.get("publication_year") == year:
            score += 0.03

        if score > best_score:
            best_score = score
            best_candidate = cand

    if best_candidate is not None and best_score >= min_similarity:
        best_candidate["_title_match_score"] = round(best_score, 4)
        return best_candidate

    return None


def enrich_records(records: List[Dict]) -> Tuple[List[Dict], Dict]:
    enriched = []
    matched_by_doi = 0
    matched_by_title = 0
    unmatched = 0
    api_errors_skipped = 0

    doi_records = [r for r in records if r.get("doi")]
    doi_list = [strip_doi_prefix(r["doi"]) for r in doi_records if r.get("doi")]
    doi_matches = {}

    for i in range(0, len(doi_list), DOI_BATCH_SIZE):
        batch = doi_list[i:i + DOI_BATCH_SIZE]
        try:
            batch_matches = bulk_lookup_by_doi(batch)
            doi_matches.update(batch_matches)
            print(f"Processed DOI batch {i} to {i + len(batch)}")
        except Exception as e:
            print(f"Skipping DOI batch {i} to {i + len(batch)} due to API error after retries: {e}")
        time.sleep(REQUEST_SLEEP_SECONDS)
    records_length = len(records)
    print(f"Processing {records_length} records")

    for i, record in enumerate(records):
        record = dict(record)
        matched_work = None
        title_match_score = None
        had_api_error = False

        doi = record.get("doi")
        if doi:
            clean_doi = strip_doi_prefix(doi)
            matched_work = doi_matches.get(clean_doi)
            if matched_work:
                matched_by_doi += 1

        if matched_work is None:
            try:
                matched_work = search_by_title(
                    title=record.get("title"),
                    year=record.get("published_year"),
                    min_similarity=TITLE_MATCH_THRESHOLD,
                )
                print(f"Processing title match for {i} of {records_length}:  {record.get('title', '')}")
                time.sleep(REQUEST_SLEEP_SECONDS)
            except Exception as e:
                had_api_error = True
                matched_work = None
                print(f"Skipping record {i} due to API error after retries: {e}")

            if matched_work is not None:
                matched_by_title += 1
                title_match_score = matched_work.get("_title_match_score")
            else:
                if had_api_error:
                    api_errors_skipped += 1
                else:
                    unmatched += 1

        if matched_work is not None:
            record["openalex_id"] = matched_work.get("id")
            record["openalex_title"] = matched_work.get("title")
            record["openalex_publication_year"] = matched_work.get("publication_year")
            record["doi"] = matched_work.get("doi") or record.get("doi")
            record["referenced_works"] = matched_work.get("referenced_works", [])
            record["cited_by_count"] = matched_work.get("cited_by_count", 0)
            record["openalex_match_status"] = "matched"
            record["openalex_title_match_score"] = title_match_score
        else:
            record["openalex_id"] = None
            record["openalex_title"] = None
            record["openalex_publication_year"] = None
            record["referenced_works"] = []
            record["cited_by_count"] = 0
            record["openalex_match_status"] = "api_error_skipped" if had_api_error else "unmatched"
            record["openalex_title_match_score"] = None

        enriched.append(record)

    metadata = {
        "source": "OpenAlex API",
        "enrichment_timestamp_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "input_records": len(records),
        "matched_by_doi": matched_by_doi,
        "matched_by_title": matched_by_title,
        "unmatched": unmatched,
        "api_errors_skipped": api_errors_skipped,
        "match_rate": round((matched_by_doi + matched_by_title) / len(records), 4) if records else 0,
        "title_match_threshold": TITLE_MATCH_THRESHOLD,
        "fields_added": [
            "openalex_id",
            "openalex_title",
            "openalex_publication_year",
            "referenced_works",
            "cited_by_count",
            "openalex_match_status",
            "openalex_title_match_score",
        ],
    }

    return enriched, metadata


def add_incoming_citations(records: List[Dict]) -> List[Dict]:
    """
    Computes incoming citations within the local corpus.
    """
    openalex_to_paper = {}
    for record in records:
        if record.get("openalex_id"):
            openalex_to_paper[record["openalex_id"]] = record["paper_id"]

    incoming_map = {record["paper_id"]: [] for record in records}

    for record in records:
        source_paper_id = record["paper_id"]
        referenced_works = record.get("referenced_works", [])

        for ref_openalex_id in referenced_works:
            target_paper_id = openalex_to_paper.get(ref_openalex_id)
            if target_paper_id is not None:
                incoming_map[target_paper_id].append(source_paper_id)

    updated_records = []
    for record in records:
        record = dict(record)
        incoming = sorted(set(incoming_map.get(record["paper_id"], [])))
        record["incoming_citations_in_corpus"] = incoming
        record["incoming_citation_count_in_corpus"] = len(incoming)
        updated_records.append(record)

    return updated_records


def add_relevant_citations(records: List[Dict]) -> List[Dict]:
    """
    Creates a combined citation relevance list for each paper.
    This merges outgoing references and incoming citations within the corpus.
    """

    # Map OpenAlex IDs to local paper IDs
    openalex_to_paper = {
        r["openalex_id"]: r["paper_id"]
        for r in records
        if r.get("openalex_id")
    }

    updated_records = []

    for record in records:
        record = dict(record)

        outgoing_in_corpus = []

        for ref in record.get("referenced_works", []):
            if ref in openalex_to_paper:
                outgoing_in_corpus.append(openalex_to_paper[ref])

        incoming = record.get("incoming_citations_in_corpus", [])

        relevant = sorted(set(outgoing_in_corpus + incoming))

        record["referenced_works_in_corpus"] = outgoing_in_corpus
        record["relevant_citations_in_corpus"] = relevant
        record["relevant_citation_count_in_corpus"] = len(relevant)

        updated_records.append(record)

    return updated_records

def add_metadata_statistics(records: List[Dict], metadata: Dict) -> Dict:
    matched_records = [r for r in records if r.get("openalex_match_status") == "matched"]

    metadata["fields_added"].extend([
        "incoming_citations_in_corpus",
        "incoming_citation_count_in_corpus",
        "relevant_citation_count_in_corpus",
    ])

    metadata["average_referenced_works"] = round(
        sum(len(r.get("referenced_works", [])) for r in matched_records) / len(matched_records), 2
    ) if matched_records else 0

    metadata["average_incoming_citations_in_corpus"] = round(
        sum(r.get("incoming_citation_count_in_corpus", 0) for r in matched_records) / len(matched_records), 2
    ) if matched_records else 0

    metadata["matched_records_with_incoming_citations"] = sum(
        1 for r in matched_records if r.get("incoming_citation_count_in_corpus", 0) > 0
    )

    return metadata


def main() -> None:
    records = load_jsonl(INPUT_JSONL)
    print(f"Loaded {len(records)} records from {INPUT_JSONL}")

    enriched_records, metadata = enrich_records(records)
    enriched_records = add_incoming_citations(enriched_records)
    enriched_records = add_relevant_citations(enriched_records)
    metadata = add_metadata_statistics(enriched_records, metadata)

    write_jsonl(enriched_records, OUTPUT_JSONL)

    with open(OUTPUT_META, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print(f"Saved enriched records to {OUTPUT_JSONL}")
    print(f"Saved enrichment metadata to {OUTPUT_META}")
    print(json.dumps(metadata, indent=2))


if __name__ == "__main__":
    main()

Loaded 15000 records from data/papers.jsonl
Processed DOI batch 0 to 100
Processed DOI batch 100 to 200
Processed DOI batch 200 to 300
Processed DOI batch 300 to 400
Processed DOI batch 400 to 500
Processed DOI batch 500 to 600
Processed DOI batch 600 to 700
Processed DOI batch 700 to 800
Processed DOI batch 800 to 900
Processed DOI batch 900 to 1000
Processed DOI batch 1000 to 1100
Processed DOI batch 1100 to 1200
Processed DOI batch 1200 to 1284
Processing 15000 records
Processing title match for 0 of 15000:  Placenta Accreta Spectrum Detection using Multimodal Deep Learning
Processing title match for 1 of 15000:  Explicit Abstention Knobs for Predictable Reliability in Video Question Answering
Processing title match for 2 of 15000:  PCEval: A Benchmark for Evaluating Physical Computing Capabilities of Large Language Models
Processing title match for 3 of 15000:  Democratizing Electronic-Photonic AI Systems: An Open-Source AI-Infused Cross-Layer Co-Design and Design Automation Toolfl